# EEG Processing

In this notebook, we assess the quality of the EEG recordings. Each EEG has different needs, so processing has to be done individually.

## 📦 Imports

In [1]:
#%matplotlib qt

from pathlib import Path 
import mindmap as mm
import mne

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `eeg`: Raw EEGs

🛠️ <font color=violet>**Configuration**</font>:
- `fconfig`: Configuration file.
- `sub`: Subject id
- `samp`: EEG sample.

In [2]:
############# Parameters ################
# configuration
fconfig = "../configurations/spike.yml" 
sub     = 3
samp    = 0
##########################################

config = mm.io.read_config("../configurations/spike.yml")
sub    = config["SUBS"][sub]
samp   = config["SAMP"][samp]
mm_dir = Path(config["MM_DIR"])

# data
data = mm.io.read_dataset(sub, "eeg")

DATA SUMMARY
Dataset: spike
Subject: sub-pat04
Retrieved objects:
- eeg
Missing objects:


In [4]:
mm.viz.plot_eeg(data["eeg"], colorscale="Turbo", title="test").show()

🌟 **Visualize Raw**

In [4]:
_ = data["eeg"].plot()
_ = data["eeg"].compute_psd().plot()

    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


## Raw
#### ⚙️ **<font color=green>Interpolate bad channels</font>** <br>
🛠️ <font color=violet>**Bad channels**</font>:
- `bads`: List of bad channels by name.

⚙️ <font color=green></font>: After observing the time series and Fourier transform, we can identify any electrodes that did not record properly. In such cases, we interpolate the faulty channels to prevent them from affecting the rest of the analysis.

In [4]:
############### Parameters ################
# bad channels
bads = ["8"]
###########################################

for b in bads:
    data["eeg"].info["bads"].append(b)
data["eeg"].interpolate_bads()

_ = data["eeg"].plot()
_ = data["eeg"].compute_psd().plot()

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 93.5 mm
Computing interpolation matrix from 203 sensor positions
Interpolating 1 sensors


/tmp/ipykernel_96418/1306104242.py:8: RuntimeWarning: (X, Y) fit (-0.8, -22.1) more than 20 mm from head frame origin
  data["eeg"].interpolate_bads()


    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


## Epoched

## Evoked

In [5]:
if config["NAME"] == "spike":
    evoked = data["eeg"].copy()

elif config["NAME"] == "intra":
    raw = data["eeg"].set_eeg_reference(["VREF"])
    evoked = raw.average()
    evoked = mne.set_eeg_reference(evoked, "average")

# plot
_ = evoked.plot()

#### ⚙️ <font color=green>**Average Referencing**</font>
We can also determine whether the reference electrode is too close to the neural sources. An ideal reference signal captures none of the brain-specific fluctuations in electric potential while picking up all of the environmental noise and interference detected by the measurement electrodes. In practice, this means that the reference electrode is often placed on the subject's body, close to the head but as far away from the neural sources as possible. If the reference electrode also records neural sources, we can set a "virtual reference," which is the average of all channels.

In [6]:
evoked.set_eeg_reference(ref_channels="average")

# plot
_ = evoked.plot()
_ = evoked.compute_psd().plot()

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


#### ⚙️ <font color=green>**Baseline correction**</font>
🛠️ <font color=violet>**Baseline**</font>:
- `baseline`: range to use as baseline.

In [9]:
############## Parameters ###############
# baseline
baseline = [0, 0.2]
#########################################

evoked.apply_baseline(baseline)
_ = evoked.plot()

Applying baseline correction (mode: mean)


#### ⚙️ <font color=green>**Notch Filtering**</font>
A prevalent source of noise in recordings from electrical devices is the 50 Hz electrical interference. To mitigate this type of noise, it is advisable to apply a notch filter, which is a band-stop filter specifically designed to attenuate frequencies around 50 Hz.

In [7]:
evoked.filter(
    l_freq = 63,
    h_freq = 37,
    picks  = "all",
    method = "fir",
)

_ = data["eeg"].plot()
_ = evoked.compute_psd().plot()

Setting up band-stop filter from 37 - 63 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 37.00
- Lower transition bandwidth: 9.25 Hz (-6 dB cutoff frequency: 32.38 Hz)
- Upper passband edge: 63.00 Hz
- Upper transition bandwidth: 15.75 Hz (-6 dB cutoff frequency: 70.88 Hz)
- Filter length: 91 samples (0.364 s)

    Using multitaper spectrum estimation with 7 DPSS windows
Plotting power spectral density (dB=True).


[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s


🌟 **Visualize Topography**

It is important to check both the EEG time series and the topological maps they generate. A topological map should be smooth; if it is not, the electrodes responsible for the irregularities are likely faulty.

In [8]:
# time points
_, half_t = mm.utils.eeg_rising_phase(evoked, 0.5)

times = [half_t/evoked.info["sfreq"]]
_ = evoked.plot_topomap(
    times = times,
    outlines = None,
    extrapolate = "local",
    size = 8,
)


💾 **Save processed EEG**

🛠️ <font color=violet>**Options**</font>:
- `overwrite`: Overwrites if file already exists.

In [9]:
################ Parameters ##############
# options
overwrite = True
##########################################

fname = mm_dir/sub/"eeg"/f"{sub}-{samp}_ave.fif"

if not fname.parent.exists():
        fname.parent.mkdir(parents=True)

evoked.save(fname, overwrite=overwrite)
print(f"\033[92mSuccessfully saved preprocessed EEG for subject '{sub}', sample '{samp}.")

Successfully saved preprocessed EEG for subject 'sub-pat04', sample 'spike.
